In [78]:
import re
import os
import shutil
import pandas as pd
import seaborn as sns
from pathlib import Path


import torch


from data_utils import *
from model import *



In [20]:
# The following is necessary if you want to use the fast tokenizer for deberta v2 or v3
# This must be done before importing transformers
trfms_path = "/home/sthueva/anaconda3/envs/kaggle/lib/python3.7/site-packages/transformers"
deberta_v2_3_tok_path = "../../input/nbroad/deberta-v2-3-fast-tokenizer"

# trfms_path = "/home/yuv4r4j/anaconda3/envs/kaggle/lib/python3.7/site-packages/transformers"
# deberta_v2_3_tok_path = "/home/yuv4r4j/workspace/input/nbroad/deberta-v2-3-fast-tokenizer"

transformers_path = Path(trfms_path)
input_dir = Path(deberta_v2_3_tok_path)

convert_file = input_dir / "convert_slow_tokenizer.py"
conversion_path = transformers_path/convert_file.name

if conversion_path.exists():
    conversion_path.unlink()

shutil.copy(convert_file, transformers_path)
deberta_v2_path = transformers_path / "models" / "deberta_v2"

for filename in ['tokenization_deberta_v2.py', 'tokenization_deberta_v2_fast.py']:
    filepath = deberta_v2_path/filename
    if filepath.exists():
        filepath.unlink()
    shutil.copy(input_dir/filename, filepath)


In [21]:
os.listdir('../../input/feedback-prize-english-language-learning')

['test.csv', 'train_fold.csv', 'train.csv', 'sample_submission.csv']

In [22]:
df = pd.read_csv('../../input/feedback-prize-english-language-learning/train_fold.csv')
print(df.shape)
df.head()

(3911, 9)


,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions,fold
0,0016926B079C,I think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0,2
1,0022683E9EA5,When a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5,0
2,00299B378633,"Dear, Principal\n\nIf u change the school poli...",3.0,3.5,3.0,3.0,3.0,2.5,1
3,003885A45F42,The best time in life is when you become yours...,4.5,4.5,4.5,4.5,4.0,5.0,3
4,0049B1DF5CCC,Small act of kindness can impact in other peop...,2.5,3.0,3.0,3.0,2.5,2.5,3


In [23]:
df['full_text'] = df['full_text'].apply(lambda x : resolve_encodings_and_normalize(x))
df['full_text'] =  df['full_text'].apply(lambda x: preprocessing(x))
df.head()

,text_id,full_text,cohesion,syntax,vocabulary,phraseology,grammar,conventions,fold
0,0016926B079C,i think that students would benefit from learn...,3.5,3.5,3.0,3.0,4.0,3.0,2
1,0022683E9EA5,when a problem is a change you have to let it ...,2.5,2.5,3.0,2.0,2.0,2.5,0
2,00299B378633,"dear, principal[br]if u change the school poli...",3.0,3.5,3.0,3.0,3.0,2.5,1
3,003885A45F42,the best time in life is when you become yours...,4.5,4.5,4.5,4.5,4.0,5.0,3
4,0049B1DF5CCC,small act of kindness can impact in other peop...,2.5,3.0,3.0,3.0,2.5,2.5,3


In [25]:
from transformers.models.deberta_v2.tokenization_deberta_v2_fast import DebertaV2TokenizerFast
tokenizer = DebertaV2TokenizerFast.from_pretrained('microsoft/deberta-v3-base', use_fast=True)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [58]:
input_ids, attention_mask = [], []
for index, text in enumerate(df.full_text):
    tokens = tokenizer(text,
                        None,
                        add_special_tokens=True,
                        truncation = True,
                        max_length=512,
                        padding = "max_length",
                        return_tensors = 'pt'
                        )                    
    input_ids.append(tokens['input_ids'])
    attention_mask.append(tokens['attention_mask'])

In [59]:
input_ids[0].shape

torch.Size([1, 512])

In [60]:
input_ids = torch.cat(input_ids)
attention_mask = torch.cat(attention_mask)
input_ids.shape, attention_mask.shape

(torch.Size([3911, 512]), torch.Size([3911, 512]))

In [70]:
syntax = torch.tensor(df["syntax"])
cohesion = torch.tensor(df["cohesion"])
vocabulary = torch.tensor(df["vocabulary"])
phraseology = torch.tensor(df["phraseology"])
grammar = torch.tensor(df["grammar"])
conventions = torch.tensor(df["conventions"])
syntax.shape, cohesion.shape, vocabulary.shape, phraseology.shape, grammar.shape, conventions.shape

(torch.Size([3911]),
 torch.Size([3911]),
 torch.Size([3911]),
 torch.Size([3911]),
 torch.Size([3911]),
 torch.Size([3911]))

In [85]:
config={
    'model' : "microsoft/deberta-v3-base",
    'model_config': None,
    'train': {"num_labels" :6}
}

In [86]:
from collections import namedtuple

In [87]:
model = Model6(namedtuple(config))

TypeError: namedtuple() missing 1 required positional argument: 'field_names'